# Fixed microenvironment simulation
Both notebooks call the same V1002 generator. All truth labels below are used only for post-training audit.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('/home/xueshuailin/CCC_Phe')
V1002_SOURCE = str(ROOT/'V1002/src')
if V1002_SOURCE in sys.path: sys.path.remove(V1002_SOURCE)
sys.path.insert(0,V1002_SOURCE)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import linear_sum_assignment
from phenoniche.v1002.lr_atlas import load_lr_atlas
from phenoniche.v1002.final_simulation import build_final_spec, simulate_final_spatial, simulate_final_bulk, bulk_potential, alr, TRUE_BETA
from phenoniche.v1002.identifiability_run import cox_fit
SEED=40700; PURITY=.5; NICHE_SIZE=100; NOISE=.05
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'


V1003 learns view-preserving deep representations, fuses them, then clusters. No prototype loss is used.

## Generate one fixed local-microenvironment dataset

In [ ]:
spec=build_final_spec(load_lr_atlas(str(ROOT/'V1002/data/commuspace_human_lr_atlas.tsv')))
sim=simulate_final_spatial(spec,purity=PURITY,niche_size=NICHE_SIZE,noise=NOISE,seed=SEED)
feature_mask=sim.final_mask.reshape(-1)
C=sim.cs
I=sim.communication[:,feature_mask]
macro_truth=np.array([0,1,1,2,3,4])[sim.labels]
assert C.shape==(1800,8) and sim.cell_types.shape==(1800,48)
assert I.shape[1]==int(feature_mask.sum()) and np.allclose(C.sum(1),1)


## Observable identifiability audit, before any model training

In [ ]:
def pair_auc(x,labels,a,b,core):
    keep=core & np.isin(labels,[a,b]); x=x[keep]; y=(labels[keep]==b).astype(int)
    cv=StratifiedKFold(3,shuffle=True,random_state=SEED)
    scores=np.empty(len(y))
    for tr,te in cv.split(x,y):
        direction=x[tr][y[tr]==1].mean(0)-x[tr][y[tr]==0].mean(0)
        scores[te]=x[te]@direction
    auc=roc_auc_score(y,scores)
    return max(auc,1-auc)
core=sim.rho>=.8
observable=pd.DataFrame([{'Pair':name,'Composition AUC':pair_auc(C,sim.labels,a,b,core),
                          'CCC AUC':pair_auc(I,sim.labels,a,b,core)}
                         for name,a,b in [('N1/N2',1,2),('N3/N4',3,4)]])
display(observable.round(3))


## Raw composition five-state sanity

In [ ]:
raw_km=KMeans(n_clusters=5,n_init=20,random_state=SEED).fit(C)
raw_pred=raw_km.labels_
raw_summary={'ARI':adjusted_rand_score(macro_truth,raw_pred),
             'silhouette':silhouette_score(C,raw_pred)}
fig,axes=plt.subplots(1,2,figsize=(10,4),constrained_layout=True)
for ax,values,title in zip(axes,[macro_truth,raw_pred],['Truth macro states','Raw C KMeans (K=5)']):
    ax.scatter(sim.coordinates[:,0],sim.coordinates[:,1],c=values,s=9,cmap='tab10',vmin=0,vmax=5)
    ax.set_title(title); ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.show()
display(pd.DataFrame([raw_summary]).round(3))


## Outcome-blind CCC filter and views

In [ ]:
display(pd.DataFrame([{'Measurable LR':len(spec.atlas),'Candidate CCC':sim.final_mask.size,
                       'Coverage passed':int(sim.coverage_mask.sum()),'Selected CCC':int(feature_mask.sum()),
                       'C shape':str(C.shape),'I shape':str(I.shape)}]))


In [ ]:
def align_clusters(pred,truth,k=6):
    counts=np.array([[(pred==i).__and__(truth==j).sum() for j in range(k)] for i in range(k)])
    rows,cols=linear_sum_assignment(-counts)
    lookup=dict(zip(rows,cols))
    return np.array([lookup[v] for v in pred]), lookup

def st_metrics(name,pred,truth=sim.labels):
    aligned,_=align_clusters(pred,truth)
    row={'Method':name,'ARI':adjusted_rand_score(truth,pred),'NMI':normalized_mutual_info_score(truth,pred)}
    row.update({label:np.mean(aligned[truth==k]==k) for k,label in enumerate(['BG','N1','N2','N3','N4','N5'])})
    return row,aligned


## Two independent view autoencoders

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
svd=TruncatedSVD(n_components=256,random_state=SEED).fit(np.log1p(I))  # ST only
scaler=StandardScaler().fit(svd.transform(np.log1p(I)))
I256=scaler.transform(svd.transform(np.log1p(I))).astype(np.float32)
class ViewAE(nn.Module):
    def __init__(self,width,hidden,latent,simplex=False):
        super().__init__(); self.encoder=nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,latent))
        self.decoder=nn.Sequential(nn.Linear(latent,hidden),nn.ReLU(),nn.Linear(hidden,width))
        self.simplex=simplex
    def forward(self,x):
        z=self.encoder(x); out=self.decoder(z)
        return z,F.softmax(out,dim=1) if self.simplex else out

def pretrain(model,x,epochs=150):
    x=torch.as_tensor(x,dtype=torch.float32,device=DEVICE); model=model.to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    for _ in range(epochs):
        opt.zero_grad(); z,reconstruction=model(x); loss=F.mse_loss(reconstruction,x); loss.backward(); opt.step()
    with torch.no_grad(): z,reconstruction=model(x)
    return model,z.detach(),reconstruction.cpu().numpy()
cae,zC,C_hat=pretrain(ViewAE(8,16,16,True),C)
iae,zI,I_hat=pretrain(ViewAE(256,128,64),I256)


## Reconstruction audit and fusion autoencoder

In [ ]:
display(pd.DataFrame([{'View':'Composition','MSE':np.mean((C-C_hat)**2),'Correlation':np.corrcoef(C.ravel(),C_hat.ravel())[0,1]},
                      {'View':'CCC SVD','MSE':np.mean((I256-I_hat)**2),'Correlation':np.corrcoef(I256.ravel(),I_hat.ravel())[0,1]}]).round(3))
u=torch.cat([F.layer_norm(zC,(16,)),F.layer_norm(zI,(64,))],dim=1).detach()
class FusionAE(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder=nn.Linear(80,64); self.decoder=nn.Linear(64,80)
    def forward(self,x):
        z=self.encoder(x); return z,self.decoder(z)
fusion=FusionAE().to(DEVICE); opt=torch.optim.Adam(fusion.parameters(),lr=1e-3)
for _ in range(200):
    opt.zero_grad(); zF,u_hat=fusion(u); loss=F.mse_loss(u_hat,u); loss.backward(); opt.step()
with torch.no_grad(): zF=fusion(u)[0].detach()


## Residual spatial refinement and post-training clustering

In [ ]:
from sklearn.neighbors import kneighbors_graph
from sklearn.preprocessing import normalize
from scipy.sparse import eye
A=kneighbors_graph(sim.coordinates,n_neighbors=12,mode='connectivity',include_self=False)
A=normalize(A.maximum(A.T)+eye(len(C)),norm='l1',axis=1)
A=torch.as_tensor(A.toarray(),dtype=torch.float32,device=DEVICE)
class SpatialRefiner(nn.Module):
    def __init__(self,d):
        super().__init__(); self.mlp=nn.Sequential(nn.Linear(d,d),nn.ReLU(),nn.Linear(d,d))
    def forward(self,z): return F.layer_norm(z+.25*self.mlp(A@z),(z.shape[1],))
def refine(z):
    z=z.detach(); model=SpatialRefiner(z.shape[1]).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=1e-3)
    target=F.layer_norm(z,(z.shape[1],)).detach()
    for _ in range(80):
        opt.zero_grad(); result=model(z); loss=F.mse_loss(result,target)+.05*F.mse_loss(result,A@target)
        loss.backward(); opt.step()
    with torch.no_grad(): return model(z).cpu().numpy()
representations={'Composition deep':refine(zC),'CCC deep':refine(zI),'C+I deep':refine(zF)}
predictions={name:KMeans(n_clusters=6,n_init=20,random_state=SEED).fit_predict(z)
             for name,z in representations.items()}


## ST maps and metrics

In [ ]:
rows=[]; aligned={}
for name,pred in predictions.items():
    row,aligned[name]=st_metrics(name,pred); rows.append(row)
raw_aligned,_=align_clusters(np.array(raw_pred),macro_truth,k=5)
rawrow={'Method':'Raw composition KMeans','ARI':adjusted_rand_score(macro_truth,raw_pred),
        'NMI':normalized_mutual_info_score(macro_truth,raw_pred)}
rawrow.update({'BG':np.mean(raw_aligned[macro_truth==0]==0),
               'N1':np.mean(raw_aligned[sim.labels==1]==1),'N2':np.mean(raw_aligned[sim.labels==2]==1),
               'N3':np.mean(raw_aligned[sim.labels==3]==2),'N4':np.mean(raw_aligned[sim.labels==4]==3),
               'N5':np.mean(raw_aligned[sim.labels==5]==4)})
display(pd.DataFrame([rawrow]+rows).round(3))
fig,axes=plt.subplots(1,4,figsize=(16,4),constrained_layout=True)
for ax,(name,values) in zip(axes,[('Truth',sim.labels),('Raw C KMeans',raw_pred),
                                  ('CCC-only',aligned['CCC deep']),('C+I deep',aligned['C+I deep'])]):
    ax.scatter(sim.coordinates[:,0],sim.coordinates[:,1],c=values,s=8,cmap='tab10',vmin=0,vmax=5)
    ax.set_title(name); ax.set_aspect('equal'); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.show()


## Soft ST assignments and pseudo-bulk bridge

In [ ]:
from scipy.special import softmax
z=representations['C+I deep']; clusters=predictions['C+I deep']
centroids=np.stack([z[clusters==k].mean(0) for k in range(6)])
q=softmax(-np.linalg.norm(z[:,None,:]-centroids[None,:,:],axis=2)/.5,axis=1)
# Aggregate cell-type expression within each sampled group; no spatial distances enter bulk potential.
anchor_sum=np.zeros((len(C),8,len(spec.genes)),dtype=np.float32)
anchor_count=np.zeros((len(C),8),dtype=np.float32)
for t in range(8):
    selected=sim.cell_types==t
    anchor_count[:,t]=selected.sum(1)
    anchor_sum[:,t]=np.einsum('am,amg->ag',selected,sim.expression,optimize=True)
rng=np.random.default_rng(SEED+20)
pseudo_C=np.empty((3000,8),dtype=np.float32)
pseudo_q=np.empty((3000,6),dtype=np.float32)
pseudo_I=np.empty((3000,256),dtype=np.float32)
for start in range(0,3000,64):
    stop=min(start+64,3000); compositions=[]; expr=[]
    for p in range(start,stop):
        selected=rng.choice(len(C),size=int(rng.integers(50,301)),replace=True)
        counts=anchor_count[selected].sum(0)
        compositions.append(counts/counts.sum())
        expr.append(anchor_sum[selected].sum(0)/np.maximum(counts[:,None],1))
        pseudo_q[p]=q[selected].mean(0)
    pseudo_C[start:stop]=np.asarray(compositions)
    potential=bulk_potential(np.asarray(expr,dtype=np.float32),spec)[:,feature_mask]
    pseudo_I[start:stop]=scaler.transform(svd.transform(np.log1p(potential)))


## Train mapper on pseudo-bulk only

In [ ]:
from sklearn.model_selection import train_test_split
x_pb=np.concatenate([pseudo_C,pseudo_I],axis=1).astype(np.float32)
tr,va=train_test_split(np.arange(3000),test_size=.2,random_state=SEED)
mapper=nn.Sequential(nn.Linear(x_pb.shape[1],256),nn.ReLU(),nn.Linear(256,128),nn.ReLU(),nn.Linear(128,6)).to(DEVICE)
opt=torch.optim.Adam(mapper.parameters(),lr=1e-3)
xtr=torch.as_tensor(x_pb[tr],device=DEVICE); ytr=torch.as_tensor(pseudo_q[tr],device=DEVICE)
for _ in range(200):
    opt.zero_grad(); logits=mapper(xtr); pred=F.softmax(logits,dim=1)
    loss=F.kl_div(F.log_softmax(logits,dim=1),ytr,reduction='batchmean')+.1*F.mse_loss(pred,ytr)
    loss.backward(); opt.step()
with torch.no_grad():
    validation=F.softmax(mapper(torch.as_tensor(x_pb[va],device=DEVICE)),dim=1).cpu().numpy()
display(pd.DataFrame([{'Pseudo-bulk validation MSE':np.mean((validation-pseudo_q[va])**2)}]).round(4))


## Shared real bulk and recovery

In [ ]:
pi_true,CB,bulk_expr,time,event,beta,eta=simulate_final_bulk(spec,sim.hc_truth,NOISE,SEED+10,patients=320)
IB=bulk_potential(bulk_expr,spec)[:,feature_mask]
assert IB.shape[1]==I.shape[1]
x_real=np.concatenate([CB,scaler.transform(svd.transform(np.log1p(IB)))],axis=1).astype(np.float32)
with torch.no_grad():
    pi_raw=F.softmax(mapper(torch.as_tensor(x_real,device=DEVICE)),dim=1).cpu().numpy()
_,lookup=align_clusters(clusters,sim.labels)
pi_pred=np.empty_like(pi_raw)
for cluster,truth_label in lookup.items(): pi_pred[:,truth_label]=pi_raw[:,cluster]
corr=[np.corrcoef(pi_true[:,k],pi_pred[:,k])[0,1] for k in range(6)]
display(pd.DataFrame({'Niche':['BG','N1','N2','N3','N4','N5'],'Bulk correlation':corr}).round(3))


## Background-reference ALR Cox

In [ ]:
cox_rows=[]
for source,pi in [('True exposure',pi_true),('Predicted exposure',pi_pred)]:
    result=cox_fit(alr(pi),time,event); gamma=np.array(result['gamma'])
    cox_rows.append({'Source':source,'Risk gamma':gamma[0],'Protective gamma':gamma[2],
                     'Neutral mean |gamma|':np.abs(gamma[[1,3,4]]).mean(),
                     'Validation C':result['validation_C'],'Test C':result['test_C']})
display(pd.DataFrame(cox_rows).round(3))


## Summary
This notebook keeps all results in memory. Review the observable audit before interpreting recovery; no automatic repair or result saving is performed.